# Master Pipeline & Reproducibility Notebook (`01_explore_data.ipynb`)

This notebook contains **all end-to-end commands and Python calls** to reproduce the entire 2-million-conversation study across **`lmsys/lmsys-chat-1m`** and **`allenai/WildChat-1M`**:

1. **Step 0:** Verify Environment & Run the 6-Case Rubric Smoke Test (`gemma4:12b`)
2. **Step 1:** Candidate Retrieval on `lmsys/lmsys-chat-1m` (Funnel A: Lexical Belief Markers + Funnel B: GPU Semantic Embeddings)
3. **Step 2:** Batch Classification of `lmsys/lmsys-chat-1m` Candidates with `gemma4:12b` (Automatic Checkpoint/Resume)
4. **Step 3:** Cross-Dataset Replication on `allenai/WildChat-1M` (Funnel A + Funnel B Retrieval & Classification)
5. **Step 4:** Generate All Statistical Reports, Stage-3 CSV Exports, and Blind Human Validation Sheets
6. **Step 5:** Verify All Generated Data Artifacts & Row Counts

> **Tip:** Long-running extraction and classification cells below use `RUN_PIPELINE = False` by default so you can safely run "Run All" to verify existing outputs without accidentally kicking off a 1-hour re-run. Set `RUN_PIPELINE = True` in any cell you want to re-execute from scratch.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

PYTHON_EXE = PROJECT_ROOT / 'venv' / 'Scripts' / 'python.exe'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

from load_data import scan_full_dataset, stream_sample
from classify import classify_conversation, run_batch, SMOKE_TEST_CASES
from analyze import analyze_results, export_stage3_and_validation_csvs, score_human_validation

print(f'Project Root : {PROJECT_ROOT}')
print(f'Python Venv  : {PYTHON_EXE} (exists={PYTHON_EXE.exists()})')

Project Root : C:\Users\cmck\confirmationBias
Python Venv  : C:\Users\cmck\confirmationBias\venv\Scripts\python.exe (exists=True)


--- 
## Step 0: Verify 3-Stage Rubric on Canonical Smoke-Test Cases

**Equivalent Terminal Command:**
```powershell
.\venv\Scripts\python.exe src\classify.py --smoke-test --no-think
```

In [2]:
RUN_SMOKE_TEST = False  # Set to True to run live against local Ollama gemma4:12b

if RUN_SMOKE_TEST:
    rows = []
    for label, messages in SMOKE_TEST_CASES:
        res = classify_conversation(messages, think=False)
        rows.append({
            'case': label,
            'prompt': messages[0]['content'][:80],
            'is_policy': res.is_policy_or_political,
            'prior_belief': res.user_states_prior_belief,
            'seeking_type': res.seeking_type,
            'reasoning': res.reasoning,
        })
    display(pd.DataFrame(rows))
else:
    print('Smoke test skipped (set RUN_SMOKE_TEST = True to execute against Ollama).')

Smoke test skipped (set RUN_SMOKE_TEST = True to execute against Ollama).


---
## Step 1: Candidate Retrieval on `lmsys/lmsys-chat-1m` (1,000,000 Conversations)

### Funnel 1A — Lexical & Declarative Belief Markers (`n = 1,021`)
**Equivalent Terminal Command:**
```powershell
.\venv\Scripts\python.exe src\load_data.py --dataset lmsys --scan-all --allow-cross-turn
```

### Funnel 1B — GPU Sentence-Embedding Semantic Filter (`n = 1,000`, no belief regex)
**Equivalent Terminal Commands:**
```powershell
ollama pull nomic-embed-text
.\venv\Scripts\python.exe src\load_data.py --dataset lmsys --scan-all --no-belief-marker --semantic-filter --semantic-top-k 1000
```

In [3]:
RUN_LMSYS_EXTRACTION = False  # Set to True to re-scan all 1,000,000 LMSYS rows

if RUN_LMSYS_EXTRACTION:
    # Funnel 1A: Lexical Belief Markers (cross-turn enabled)
    scan_full_dataset(
        dataset_name='lmsys/lmsys-chat-1m',
        allow_cross_turn=True,
        require_belief_marker=True,
    )
    # Funnel 1B: GPU Semantic-Embedding Filter (no belief regex required)
    scan_full_dataset(
        dataset_name='lmsys/lmsys-chat-1m',
        require_belief_marker=False,
        semantic_filter=True,
        semantic_top_k=1000,
        embed_model='nomic-embed-text',
    )
else:
    print('LMSYS extraction already completed:')
    print(' -', RAW_DIR / 'candidates_full_1m_belief_crossturn.parquet')
    print(' -', RAW_DIR / 'candidates_full_1m_all_policy_sem1000.parquet')

LMSYS extraction already completed:
 - C:\Users\cmck\confirmationBias\data\raw\candidates_full_1m_belief_crossturn.parquet
 - C:\Users\cmck\confirmationBias\data\raw\candidates_full_1m_all_policy_sem1000.parquet


---
## Step 2: Classify `lmsys/lmsys-chat-1m` Candidates with `gemma4:12b`

`run_batch()` automatically checkpoints every row to `.jsonl` and resumes where it left off if interrupted.

**Equivalent Terminal Commands:**
```powershell
# Classify Funnel 1A (1,021 candidates)
.\venv\Scripts\python.exe src\classify.py --input data\raw\candidates_full_1m_belief_crossturn.parquet

# Classify Funnel 1B (1,000 semantic-filtered candidates)
.\venv\Scripts\python.exe src\classify.py --input data\raw\candidates_full_1m_all_policy_sem1000.parquet
```

In [ ]:
RUN_LMSYS_CLASSIFICATION = False  # Set to True to run/resume classification via Ollama

if RUN_LMSYS_CLASSIFICATION:
    run_batch(RAW_DIR / 'candidates_full_1m_belief_crossturn.parquet')
    run_batch(RAW_DIR / 'candidates_full_1m_all_policy_sem1000.parquet')
else:
    print('LMSYS classifications already completed:')
    print(' -', PROCESSED_DIR / 'classified_candidates_full_1m_belief_crossturn.jsonl')
    print(' -', PROCESSED_DIR / 'classified_candidates_full_1m_all_policy_sem1000.jsonl')

LMSYS classifications already completed:
 - C:\Users\cmck\confirmationBias\data\processed\classified_candidates_full_1m_belief_crossturn.jsonl
 - C:\Users\cmck\confirmationBias\data\processed\classified_candidates_full_1m_all_policy_sem1000.jsonl


---
## Step 3: Cross-Dataset Replication on `allenai/WildChat-1M` (1,000,000 Conversations)

Runs the exact same dual-funnel retrieval and `gemma4:12b` classification on `allenai/WildChat-1M`.

**Equivalent Terminal Commands:**
```powershell
# 3A. Extract WildChat Funnel A (Lexical Belief Markers) & Funnel B (GPU Semantic Filter)
.\venv\Scripts\python.exe src\load_data.py --dataset wildchat --scan-all --allow-cross-turn
.\venv\Scripts\python.exe src\load_data.py --dataset wildchat --scan-all --no-belief-marker --semantic-filter --semantic-top-k 1000

# 3B. Classify both WildChat candidate sets with gemma4:12b
.\venv\Scripts\python.exe src\classify.py --input data\raw\wildchat_candidates_full_1m_belief_crossturn.parquet
.\venv\Scripts\python.exe src\classify.py --input data\raw\wildchat_candidates_full_1m_all_policy_sem1000.parquet
```

In [ ]:
RUN_WILDCHAT_PIPELINE = False  # Set to True to re-run WildChat-1M extraction and classification

if RUN_WILDCHAT_PIPELINE:
    scan_full_dataset(dataset_name='allenai/WildChat-1M', allow_cross_turn=True, require_belief_marker=True)
    scan_full_dataset(
        dataset_name='allenai/WildChat-1M',
        require_belief_marker=False,
        semantic_filter=True,
        semantic_top_k=1000,
        embed_model='nomic-embed-text',
    )
    run_batch(RAW_DIR / 'wildchat_candidates_full_1m_belief_crossturn.parquet')
    run_batch(RAW_DIR / 'wildchat_candidates_full_1m_all_policy_sem1000.parquet')
else:
    print('WildChat-1M classifications already completed:')
    print(' -', PROCESSED_DIR / 'classified_wildchat_candidates_full_1m_belief_crossturn.jsonl')
    print(' -', PROCESSED_DIR / 'classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl')

WildChat-1M classifications already completed:
 - C:\Users\cmck\confirmationBias\data\processed\classified_wildchat_candidates_full_1m_belief_crossturn.jsonl
 - C:\Users\cmck\confirmationBias\data\processed\classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl


---
## Step 4: Generate All Statistical Reports & Export Stage-3 / Validation CSVs

**Equivalent Terminal Commands:**
```powershell
# 1. LMSYS Funnel A only (Explicit Belief Markers)
.\venv\Scripts\python.exe src\analyze.py --input data\processed\classified_candidates_full_1m_belief_crossturn.jsonl --output-report data\processed\statistical_report.txt

# 2. LMSYS Funnel B only (GPU Semantic Filter)
.\venv\Scripts\python.exe src\analyze.py --input data\processed\classified_candidates_full_1m_all_policy_sem1000.jsonl --output-report data\processed\statistical_report_sem1000.txt

# 3. LMSYS Combined (1M Funnel A + B)
.\venv\Scripts\python.exe src\analyze.py --input data\processed\classified_candidates_full_1m_belief_crossturn.jsonl data\processed\classified_candidates_full_1m_all_policy_sem1000.jsonl --output-report data\processed\statistical_report_combined.txt

# 4. WildChat Combined (1M Funnel A + B)
.\venv\Scripts\python.exe src\analyze.py --input data\processed\classified_wildchat_candidates_full_1m_belief_crossturn.jsonl data\processed\classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl --output-report data\processed\statistical_report_wildchat.txt

# 5. Grand 2-Million-Conversation Pooled Report + Export All 455 Stage-3 Conversations to CSV
.\venv\Scripts\python.exe src\analyze.py --input data\processed\classified_candidates_full_1m_belief_crossturn.jsonl data\processed\classified_candidates_full_1m_all_policy_sem1000.jsonl data\processed\classified_wildchat_candidates_full_1m_belief_crossturn.jsonl data\processed\classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl --output-report data\processed\statistical_report_2m_pooled.txt --export-csv

# 6. Score Human Validation (Cohen's kappa) once human_validation_blind_sample_60.csv is labeled
.\venv\Scripts\python.exe src\analyze.py --score-validation
```

In [6]:
# Run the Grand 2-Million-Conversation Pooled Statistical Report directly in Python
all_jsonl_files = [
    PROCESSED_DIR / 'classified_candidates_full_1m_belief_crossturn.jsonl',
    PROCESSED_DIR / 'classified_candidates_full_1m_all_policy_sem1000.jsonl',
    PROCESSED_DIR / 'classified_wildchat_candidates_full_1m_belief_crossturn.jsonl',
    PROCESSED_DIR / 'classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl',
]

report_text = analyze_results(all_jsonl_files)
print(report_text)

CONFIRMATION VS. FALSIFICATION INFORMATION-SEEKING: STATISTICAL REPORT
Input file(s): C:\Users\cmck\confirmationBias\data\processed\classified_candidates_full_1m_belief_crossturn.jsonl, C:\Users\cmck\confirmationBias\data\processed\classified_candidates_full_1m_all_policy_sem1000.jsonl, C:\Users\cmck\confirmationBias\data\processed\classified_wildchat_candidates_full_1m_belief_crossturn.jsonl, C:\Users\cmck\confirmationBias\data\processed\classified_wildchat_candidates_full_1m_all_policy_sem1000.jsonl

1. HIERARCHICAL CLASSIFICATION FUNNEL
------------------------------------------------------------------------------
  Total classified conversations          : 3,489
  Stage 1: Policy / Political topic       : 1,631 (46.7%)
  Stage 2: User states prior belief       : 537 (32.9% of policy)
  Stage 3: Information-seeking query      : 455 (84.7% of belief)

2. STAGE 3 PREVALENCE BREAKDOWN (AMONG INFORMATION-SEEKING QUERIES, N = 455)
---------------------------------------------------------

---
## Step 5: Inventory & Verification of All Generated Data Artifacts

In [7]:
artifact_rows = []
for folder_label, folder in [('data/raw', RAW_DIR), ('data/processed', PROCESSED_DIR)]:
    for p in sorted(folder.iterdir()):
        if p.is_file():
            size_kb = round(p.stat().st_size / 1024, 1)
            row_count = ''
            if p.suffix == '.parquet':
                row_count = f'{len(pd.read_parquet(p)):,} rows'
            elif p.suffix == '.jsonl':
                with open(p, encoding='utf-8') as f:
                    row_count = f'{sum(1 for line in f if line.strip()):,} rows'
            elif p.suffix == '.csv':
                row_count = f'{len(pd.read_csv(p)):,} rows'
            artifact_rows.append({
                'Directory': folder_label,
                'File Name': p.name,
                'Rows / Records': row_count,
                'Size (KB)': size_kb,
            })

pd.DataFrame(artifact_rows)

,Directory,File Name,Rows / Records,Size (KB)
0,data/raw,candidates_full_1m_all_policy_sem1000.parquet,"1,000 rows",2084.7
1,data/raw,candidates_full_1m_belief.parquet,630 rows,1642.9
2,data/raw,candidates_full_1m_belief_crossturn.parquet,"1,021 rows",3954.0
3,data/raw,sample_english_100.parquet,100 rows,175.6
4,data/raw,sample_english_policy_500.parquet,500 rows,897.4
5,data/raw,sample_english_policy_belief_100.parquet,100 rows,330.0
6,data/raw,sample_english_policy_belief_500.parquet,500 rows,2058.5
7,data/raw,wildchat_candidates_full_1m_all_policy_sem1000.parquet,"1,000 rows",3729.8
8,data/raw,wildchat_candidates_full_1m_belief_crossturn.parquet,651 rows,4541.7
9,data/raw,wildchat_sample_english_policy_belief_200.parquet,200 rows,1005.1
